# Retail Sales Data Processing and Business Insights Generation
## Project Objective

The objective of this project is to build an end-to-end retail data engineering pipeline capable of ingesting, cleaning, transforming, validating, and analyzing retail sales transaction data. The project also aims to generate business insights and create an interactive Power BI dashboard for reporting and decision-making.


In [3]:
import pandas as pd
import numpy as np
import os

## Data Ingestion

In [4]:
# Load Excel File
file_path = r"../Dataset/USECASE - Data Engineering.xlsx"

# Read Sheets
product_details = pd.read_excel(file_path, sheet_name='product_details')
retail_data1 = pd.read_excel(file_path, sheet_name='retail_data1')
retail_data2 = pd.read_excel(file_path, sheet_name='retail_data2')

print(product_details.head())
print(retail_data1.head())
print(retail_data2.head())

   product_id product_name     category   price
0         101       Laptop  Electronics  250000
1         102        Phone  Electronics   70000
2         103        Shirt     Clothing    1300
3         104        Shoes     Clothing    8000
4         105           TV  Electronics   45000
   transaction_id  customer_id   customer_name  product_id    price  \
0               1          642   Troy Mitchell         108  15000.0   
1               2          881  Sarah Guerrero         102  70000.0   
2               3          505   Samantha Hull         109  65000.0   
3               4          794   Gerald Cooper         106  80000.0   
4               5          864   Cameron Black         108  15000.0   

    product_name         category purchase_location       city  \
0  Mixer grinder  Home Appliances           offline  Bangalore   
1          Phone             ELEC            online  Bangalore   
2   Refrigerator  Home Appliances           offline    Chennai   
3           sofa     

## Data Profiling and Quality Checks


In [5]:
# Combine Datasets
retail_data = pd.concat([retail_data1, retail_data2], ignore_index=True)

# Missing Values
print(retail_data.isnull().sum())

# Duplicate Records
print(retail_data.duplicated().sum())

# Unique Product Names
print(retail_data['product_name'].unique())

# Unique Categories
print(retail_data['category'].unique())

transaction_id         0
customer_id            0
customer_name          0
product_id             0
price                809
product_name           0
category               0
purchase_location      0
city                   0
transaction_date       0
quantity               0
payment_method         0
discount               0
email                  0
phone                  0
payment_status         0
dtype: int64
0
<StringArray>
['Mixer grinder',         'Phone',  'Refrigerator',          'sofa',
        'laptop',         'phone',     'Microwave',         'PHONE',
         'Shoes',          'SOFA',  'REFRIGERATOR',  'refrigerator',
         'SHOES',         'SHIRT',            'Tv',  'DINING TABLE',
  'dining table',     'MICROWAVE',         'Shirt',         'shirt',
            'TV',         'shoes',        'Laptop',            'tv',
 'MIXER GRINDER',     'microwave', 'mixer grinder',  'Dining table',
          'Sofa',        'LAPTOP']
Length: 30, dtype: str
<StringArray>
['Home Appliance

## Data Cleaning and Transformation

In [6]:
# Standardize Product Names
retail_data['product_name'] = retail_data['product_name'].str.strip().str.title()

# Category Mapping
category_mapping = {
    'ELEC': 'Electronics',
    'electronics': 'Electronics',
    'Electronics': 'Electronics',

    'CLOTH': 'Clothing',
    'clothing': 'Clothing',
    'Clothing': 'Clothing',

    'FURN': 'Furniture',
    'furniture': 'Furniture',
    'Furniture': 'Furniture',

    'HOME': 'Home Appliances',
    'home appliances': 'Home Appliances',
    'Home Appliances': 'Home Appliances'
}

retail_data['category'] = retail_data['category'].replace(category_mapping)

# Fill Missing Prices
price_mapping = product_details.set_index('product_id')['price']

retail_data['price'] = retail_data['price'].fillna(
    retail_data['product_id'].map(price_mapping)
)

# Remove Invalid Quantities
retail_data = retail_data[retail_data['quantity'] > 0]

# Convert Dates
retail_data['transaction_date'] = pd.to_datetime(
    retail_data['transaction_date'],
    errors='coerce'
)

## PII Masking

In [7]:
# Mask Email Function
def mask_email(email):
    email = str(email)

    if '@' in email:
        parts = email.split('@')
        name = parts[0]
        domain = parts[1]

        if len(name) > 3:
            masked_name = name[:3] + '*' * (len(name) - 3)
        else:
            masked_name = '*' * len(name)

        return masked_name + '@' + domain

    return email

# Mask Phone Function
def mask_phone(phone):
    phone = str(phone)

    if len(phone) >= 4:
        return phone[:2] + '*' * 6 + phone[-2:]

    return phone

# Apply Masking
retail_data['masked_email'] = retail_data['email'].apply(mask_email)
retail_data['masked_phone'] = retail_data['phone'].apply(mask_phone)

# Display Sample
retail_data[['email', 'masked_email', 'phone', 'masked_phone']].head()

,email,masked_email,phone,masked_phone
0,Troy60@gmail.com,Tro***@gmail.com,8385276968,83******68
1,SarahGuerrero@yahoo.com,Sar**********@yahoo.com,7147248911,71******11
2,SamanthaHull@outlook.com,Sam*********@outlook.com,7415321565,74******65
3,GeraldCooper@gmail.com,Ger*********@gmail.com,9739354201,97******01
4,CameronBlack@yahoo.com,Cam*********@yahoo.com,9938169477,99******77


## KPI Generation and Business Insights

In [8]:
# Revenue Calculation
retail_data['revenue'] = (
    retail_data['price'] *
    retail_data['quantity']
) - retail_data['discount']

# Total Revenue
total_revenue = retail_data['revenue'].sum()

print("TOTAL REVENUE")
print(total_revenue)

# Revenue by Category
revenue_by_category = retail_data.groupby('category')['revenue'].sum()

print("\nREVENUE BY CATEGORY")
print(revenue_by_category)

# Revenue by City
revenue_by_city = retail_data.groupby('city')['revenue'].sum()

print("\nREVENUE BY CITY")
print(revenue_by_city)

# Top Selling Products
top_products = retail_data.groupby('product_name')['quantity'].sum().sort_values(ascending=False)

print("\nTOP SELLING PRODUCTS")
print(top_products)

# Payment Methods
payment_method = retail_data['payment_method'].value_counts()

print("\nPAYMENT METHOD DISTRIBUTION")
print(payment_method)

TOTAL REVENUE
1598240114.95

REVENUE BY CATEGORY
category
Clothing           2.327163e+07
Electronics        9.323244e+08
Furniture          3.313896e+08
Home Appliances    3.112544e+08
Name: revenue, dtype: float64

REVENUE BY CITY
city
Bangalore    2.985489e+08
Chennai      3.343395e+08
Delhi        3.292955e+08
Hyderabad    3.199222e+08
Mumbai       3.161339e+08
Name: revenue, dtype: float64

TOP SELLING PRODUCTS
product_name
Laptop           2585
Dining Table     2583
Microwave        2550
Sofa             2528
Shoes            2519
Phone            2488
Tv               2487
Mixer Grinder    2475
Refrigerator     2452
Shirt            2400
Name: quantity, dtype: int64

PAYMENT METHOD DISTRIBUTION
payment_method
UPI           2139
Cash          2123
Card          2121
NetBanking    2018
Name: count, dtype: int64


## Exporting Cleaned Dataset

In [10]:
# Create Output Folder
os.makedirs("../Output", exist_ok=True)

# Export Dataset
output_path = r"../Output/final_cleaned_retail_data.csv"

retail_data.to_csv(output_path, index=False)

print("Dataset Exported Successfully")

Dataset Exported Successfully


## Conclusion

The retail sales data engineering pipeline was successfully implemented using Python and Pandas. The project included data ingestion, profiling, data cleaning, transformation, PII masking, KPI generation, and exporting a curated dataset for Power BI reporting and analytics.